# Amazon ML Challenge — Dataset Reconnaissance

## Objective

Understand the structure, scale, quality, and matching distribution of the
three business entity sources before building the entity-resolution pipeline.

### Sources

- S1 — Reference / deduplicated entities
- S2 — Noisy business records
- S3 — Noisy business records
- Ground Truth — Known S1 → S2/S3 matches for training

### Day 1 Goals

1. Verify dataset structure and schemas
2. Measure dataset sizes
3. Analyze missing values
4. Analyze duplicate records
5. Analyze country distributions
6. Analyze ground-truth match distribution
7. Inspect unmatched, single-match, and multi-match entities
8. Prepare the entity-level validation setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("pandas version:", pd.__version__)

pandas version: 3.0.6


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..")
DATASET_DIR = PROJECT_ROOT / "dataset"

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

TRAIN_FILES = {
    "S1": TRAIN_DIR / "train_source1.tsv",
    "S2": TRAIN_DIR / "train_source2.tsv",
    "S3": TRAIN_DIR / "train_source3.tsv",
    "GT": TRAIN_DIR / "train_ground_truth.tsv",
}

TEST_FILES = {
    "S1": TEST_DIR / "test_source1.tsv",
    "S2": TEST_DIR / "test_source2.tsv",
    "S3": TEST_DIR / "test_source3.tsv",
}

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset directory:", DATASET_DIR.resolve())

print("\nTraining files:")
for name, path in TRAIN_FILES.items():
    print(f"{name}: {path} | exists={path.exists()}")

print("\nTest files:")
for name, path in TEST_FILES.items():
    print(f"{name}: {path} | exists={path.exists()}")

Project root: C:\Users\sneha\Amazon-ML-Challenge-2026\student_resource
Dataset directory: C:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\dataset

Training files:
S1: ..\dataset\train\train_source1.tsv | exists=True
S2: ..\dataset\train\train_source2.tsv | exists=True
S3: ..\dataset\train\train_source3.tsv | exists=True
GT: ..\dataset\train\train_ground_truth.tsv | exists=True

Test files:
S1: ..\dataset\test\test_source1.tsv | exists=True
S2: ..\dataset\test\test_source2.tsv | exists=True
S3: ..\dataset\test\test_source3.tsv | exists=True


In [3]:
def file_size_mb(path):
    return path.stat().st_size / (1024 ** 2)

print("TRAIN DATA")
print("-" * 50)

for name, path in TRAIN_FILES.items():
    print(f"{name}: {file_size_mb(path):,.2f} MB")

print("\nTEST DATA")
print("-" * 50)

for name, path in TEST_FILES.items():
    print(f"{name}: {file_size_mb(path):,.2f} MB")

TRAIN DATA
--------------------------------------------------
S1: 200.34 MB
S2: 466.63 MB
S3: 480.37 MB
GT: 121.13 MB

TEST DATA
--------------------------------------------------
S1: 166.91 MB
S2: 485.86 MB
S3: 482.56 MB


In [4]:
def inspect_file(path, nrows=5):
    df = pd.read_csv(
        path,
        sep="\t",
        nrows=nrows
    )
    
    print(f"\nFile: {path.name}")
    print("Shape of sample:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head())

for name, path in TRAIN_FILES.items():
    inspect_file(path)


File: train_source1.tsv
Shape of sample: (5, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India



File: train_source2.tsv
Shape of sample: (5, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



File: train_source3.tsv
Shape of sample: (5, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block Jayanagar, Bengaluru Urban, Bangalore, ಕರ್ನಾಟಕ",India



File: train_ground_truth.tsv
Shape of sample: (5, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785"


In [5]:
EXPECTED_SOURCE_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country"
]

EXPECTED_GT_COLUMNS = [
    "source1_entity_id",
    "matched_entity_ids"
]

for name in ["S1", "S2", "S3"]:
    df_sample = pd.read_csv(
        TRAIN_FILES[name],
        sep="\t",
        nrows=1
    )
    
    assert list(df_sample.columns) == EXPECTED_SOURCE_COLUMNS, (
        f"{name} schema mismatch: {list(df_sample.columns)}"
    )

gt_sample = pd.read_csv(
    TRAIN_FILES["GT"],
    sep="\t",
    nrows=1
)

assert list(gt_sample.columns) == EXPECTED_GT_COLUMNS, (
    f"Ground truth schema mismatch: {list(gt_sample.columns)}"
)

print("✓ All training schemas match the expected structure.")

✓ All training schemas match the expected structure.


## 2. Dataset Size Audit

Because the source files are large, the full datasets will be processed in chunks rather than loaded into memory simultaneously.

In [6]:
CHUNK_SIZE = 100_000

def count_rows(path, chunksize=CHUNK_SIZE):
    total = 0
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize,
        usecols=[0]
    ):
        total += len(chunk)
    
    return total


row_counts = {}

for name, path in TRAIN_FILES.items():
    print(f"Counting {name}...")
    row_counts[name] = count_rows(path)

print("\nTraining row counts:")
for name, count in row_counts.items():
    print(f"{name}: {count:,}")

Counting S1...
Counting S2...
Counting S3...
Counting GT...

Training row counts:
S1: 2,206,821
S2: 5,034,616
S3: 5,285,603
GT: 2,206,821


In [7]:
test_row_counts = {}

for name, path in TEST_FILES.items():
    print(f"Counting test {name}...")
    test_row_counts[name] = count_rows(path)

print("\nTest row counts:")
for name, count in test_row_counts.items():
    print(f"{name}: {count:,}")

Counting test S1...
Counting test S2...
Counting test S3...

Test row counts:
S1: 1,732,544
S2: 4,887,273
S3: 5,082,316


## 3. Missing-Value Audit

We inspect missing values in the source fields without loading the complete
datasets into memory.

This will help us understand how frequently business names, addresses, and
country values are unavailable.

In [8]:
def missing_value_audit(path, chunksize=CHUNK_SIZE):
    total_rows = 0
    missing_counts = None

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        total_rows += len(chunk)

        current_missing = chunk.isna().sum()

        if missing_counts is None:
            missing_counts = current_missing
        else:
            missing_counts = missing_counts.add(
                current_missing,
                fill_value=0
            )

    result = pd.DataFrame({
        "missing_count": missing_counts.astype(int),
        "missing_percentage": (
            missing_counts / total_rows * 100
        ).round(4)
    })

    return result


for name in ["S1", "S2", "S3"]:
    print(f"\n{'=' * 60}")
    print(f"TRAIN {name}")
    print("=" * 60)

    audit = missing_value_audit(TRAIN_FILES[name])
    display(audit)


TRAIN S1


,missing_count,missing_percentage
entity_id,0,0.0
business_name,0,0.0
business_address,0,0.0
country,0,0.0



TRAIN S2


,missing_count,missing_percentage
entity_id,0,0.0000
business_name,2,0.0000
business_address,168967,3.3561
country,0,0.0000



TRAIN S3


,missing_count,missing_percentage
entity_id,0,0.0000
business_name,13,0.0002
business_address,175916,3.3282
country,0,0.0000


## 4. Duplicate Audit

Check whether entity IDs are unique within each source and whether complete
business records are duplicated.

Duplicate analysis is performed in chunks because the source files contain
millions of records.

In [9]:
def duplicate_id_audit(path, chunksize=CHUNK_SIZE):
    total_rows = 0
    unique_ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize,
        usecols=["entity_id"]
    ):
        total_rows += len(chunk)
        unique_ids.update(chunk["entity_id"].dropna())

    unique_count = len(unique_ids)
    duplicate_count = total_rows - unique_count

    return {
        "total_rows": total_rows,
        "unique_entity_ids": unique_count,
        "duplicate_entity_ids": duplicate_count
    }


duplicate_results = {}

for name in ["S1", "S2", "S3"]:
    print(f"Auditing {name}...")
    duplicate_results[name] = duplicate_id_audit(
        TRAIN_FILES[name]
    )

duplicate_results_df = pd.DataFrame(duplicate_results).T

display(duplicate_results_df)

Auditing S1...
Auditing S2...
Auditing S3...


,total_rows,unique_entity_ids,duplicate_entity_ids
S1,2206821,2206821,0
S2,5034616,5034616,0
S3,5285603,5285603,0


## 5. Country Distribution

Analyze the country distribution in each training source.

Country will be treated as an open-set categorical field. We will not hard-code
the model or pipeline to only the countries observed in training.

In [10]:
def country_audit(path, chunksize=CHUNK_SIZE):
    country_counts = {}

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize,
        usecols=["country"]
    ):
        counts = chunk["country"].value_counts(dropna=False)

        for country, count in counts.items():
            country_counts[country] = (
                country_counts.get(country, 0) + count
            )

    result = pd.DataFrame(
        list(country_counts.items()),
        columns=["country", "count"]
    )

    result["percentage"] = (
        result["count"] /
        result["count"].sum() *
        100
    ).round(4)

    return result.sort_values(
        "count",
        ascending=False
    ).reset_index(drop=True)


country_distributions = {}

for name in ["S1", "S2", "S3"]:
    print(f"\n{'=' * 60}")
    print(f"TRAIN {name}")
    print("=" * 60)

    country_distributions[name] = country_audit(
        TRAIN_FILES[name]
    )

    display(country_distributions[name])


TRAIN S1


,country,count,percentage
0,US,1323633,59.9792
1,India,883188,40.0208



TRAIN S2


,country,count,percentage
0,US,3016817,59.9215
1,India,2017799,40.0785



TRAIN S3


,country,count,percentage
0,US,3170056,59.9753
1,India,2115547,40.0247


## 6. Ground-Truth Match Distribution

The ground-truth file contains one row for every training S1 entity.

`matched_entity_ids` contains a comma-separated list of matching S2/S3
entity IDs and is empty when the S1 entity has no match.

We calculate the number of true matches for every S1 entity and analyze the
distribution of zero, single, and multiple matches.

In [11]:
def ground_truth_match_distribution(
    path,
    chunksize=CHUNK_SIZE
):
    match_count_distribution = {}
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize,
        keep_default_na=False
    ):
        total_rows += len(chunk)

        match_counts = chunk["matched_entity_ids"].apply(
            lambda x: 0 if x.strip() == "" else len(x.split(","))
        )

        counts = match_counts.value_counts()

        for match_count, frequency in counts.items():
            match_count_distribution[match_count] = (
                match_count_distribution.get(match_count, 0)
                + frequency
            )

    result = pd.DataFrame(
        list(match_count_distribution.items()),
        columns=["num_matches", "num_s1_entities"]
    )

    result["percentage"] = (
        result["num_s1_entities"]
        / total_rows
        * 100
    ).round(4)

    return result.sort_values(
        "num_matches"
    ).reset_index(drop=True)


gt_distribution = ground_truth_match_distribution(
    TRAIN_FILES["GT"]
)

display(gt_distribution)

,num_matches,num_s1_entities,percentage
0,0,123247,5.5848
1,1,119157,5.3995
2,2,375212,17.0024
3,3,530841,24.0546
4,4,484115,21.9372
5,5,321957,14.5892
6,6,164868,7.4708
7,7,63968,2.8986
8,8,18680,0.8465
9,9,4205,0.1905


## 7. Ground-Truth Examples

Inspect representative S1 entities with:

- zero matches
- exactly one match
- multiple matches

This provides concrete examples of the entity-resolution task before
constructing the validation set and matcher.

In [12]:
# Read only the ground truth file.
# We need the examples themselves, so this is intentionally kept separate
# from the aggregate chunk-based audits.

gt_sample = pd.read_csv(
    TRAIN_FILES["GT"],
    sep="\t",
    keep_default_na=False
)

print("Ground truth rows:", len(gt_sample))

Ground truth rows: 2206821


In [13]:
# Number of matches for every S1
gt_sample["num_matches"] = gt_sample["matched_entity_ids"].apply(
    lambda x: 0 if x.strip() == "" else len(x.split(","))
)

print("Unmatched examples:")
display(
    gt_sample[gt_sample["num_matches"] == 0]
    .head(3)
)

print("\nSingle-match examples:")
display(
    gt_sample[gt_sample["num_matches"] == 1]
    .head(3)
)

print("\nMulti-match examples:")
display(
    gt_sample[gt_sample["num_matches"] >= 5]
    .head(3)
)

Unmatched examples:


,source1_entity_id,matched_entity_ids,num_matches
21,S1-302869473,,0
37,S1-262997549,,0
40,S1-508022910,,0



Single-match examples:


,source1_entity_id,matched_entity_ids,num_matches
34,S1-116043204,S3-85523430,1
41,S1-473377609,S3-433876173,1
49,S1-439203009,S3-967165288,1



Multi-match examples:


,source1_entity_id,matched_entity_ids,num_matches
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364",5
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785",6
12,S1-546142636,"S2-487600131,S2-582477216,S2-392804085,S3-200008747,S3-729771680,S3-249331830",6


## 8. Inspect S1 ↔ S2/S3 Matching Examples

Retrieve the actual records corresponding to selected ground-truth
relationships without loading the full S2/S3 datasets into memory.

This allows us to observe real name and address variation patterns.

In [14]:
def retrieve_records_by_id(path, entity_ids, chunksize=CHUNK_SIZE):
    """
    Retrieve only requested entity IDs from a large TSV file.
    """
    entity_ids = set(entity_ids)
    results = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        matched = chunk[chunk["entity_id"].isin(entity_ids)]

        if not matched.empty:
            results.append(matched)

        if sum(len(x) for x in results) >= len(entity_ids):
            break

    if results:
        return pd.concat(results, ignore_index=True)

    return pd.DataFrame(
        columns=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    )

In [15]:
# Select one example from each category

unmatched_id = (
    gt_sample.loc[
        gt_sample["num_matches"] == 0,
        "source1_entity_id"
    ]
    .iloc[0]
)

single_match_row = (
    gt_sample[gt_sample["num_matches"] == 1]
    .iloc[0]
)

multi_match_row = (
    gt_sample[gt_sample["num_matches"] >= 5]
    .iloc[0]
)

single_id = single_match_row["source1_entity_id"]
single_match_ids = single_match_row["matched_entity_ids"].split(",")

multi_id = multi_match_row["source1_entity_id"]
multi_match_ids = multi_match_row["matched_entity_ids"].split(",")

print("Unmatched S1:", unmatched_id)
print("Single-match S1:", single_id)
print("  Match:", single_match_ids)
print("Multi-match S1:", multi_id)
print("  Matches:", multi_match_ids)

Unmatched S1: S1-302869473
Single-match S1: S1-116043204
  Match: ['S3-85523430']
Multi-match S1: S1-965667
  Matches: ['S2-681193310', 'S2-743505751', 'S3-775321672', 'S3-11291185', 'S3-860443364']


In [16]:
# Collect all IDs we need from each source

s1_ids = {
    unmatched_id,
    single_id,
    multi_id
}

s2_ids = set()
s3_ids = set()

for entity_id in single_match_ids + multi_match_ids:
    if entity_id.startswith("S2-"):
        s2_ids.add(entity_id)
    elif entity_id.startswith("S3-"):
        s3_ids.add(entity_id)

# Retrieve records
s1_examples = retrieve_records_by_id(
    TRAIN_FILES["S1"],
    s1_ids
)

s2_examples = retrieve_records_by_id(
    TRAIN_FILES["S2"],
    s2_ids
)

s3_examples = retrieve_records_by_id(
    TRAIN_FILES["S3"],
    s3_ids
)

print("S1 records retrieved:", len(s1_examples))
print("S2 records retrieved:", len(s2_examples))
print("S3 records retrieved:", len(s3_examples))

S1 records retrieved: 3
S2 records retrieved: 2
S3 records retrieved: 4


In [17]:
print("S1 examples")
display(s1_examples)

print("S2 matching examples")
display(s2_examples)

print("S3 matching examples")
display(s3_examples)

S1 examples


,entity_id,business_name,business_address,country
0,S1-302869473,International Automation Consultants Inc,"329 Rev Walton Drive, Lockport, IL",US
1,S1-116043204,Red Consultants Pvt. Ltd.,"C/O Tapas Kumar Betal, Bhogpur, Purba Medinipur, Panskura, East Midnapore, West Bengal",India
2,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


S2 matching examples


,entity_id,business_name,business_address,country
0,S2-681193310,Maure Wilblims Colombier Inc,NaN,US
1,S2-743505751,Maure Williams Colombier,NaN,US


S3 matching examples


,entity_id,business_name,business_address,country
0,S3-860443364,Maure Williams Inc Center,NaN,US
1,S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US
2,S3-85523430,Red Pvt. Ltd. Center,NaN,India
3,S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US


## 9. Entity-Level Validation Split

The test set does not contain ground-truth labels, so model development will
use a held-out validation set derived from the training entities.

The split is performed at the S1 entity level so that all candidate pairs
belonging to an S1 entity remain in the same partition.

This prevents information leakage between training and validation.

In [18]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

s1_ids = gt_sample["source1_entity_id"].values

train_s1_ids, valid_s1_ids = train_test_split(
    s1_ids,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE
)

print("Total S1 entities:", len(s1_ids))
print("Training S1 entities:", len(train_s1_ids))
print("Validation S1 entities:", len(valid_s1_ids))

print(
    "\nValidation percentage:",
    round(len(valid_s1_ids) / len(s1_ids) * 100, 2),
    "%"
)

Total S1 entities: 2206821
Training S1 entities: 1765456
Validation S1 entities: 441365

Validation percentage: 20.0 %


In [19]:
train_s1_set = set(train_s1_ids)
valid_s1_set = set(valid_s1_ids)

print("Overlap:", len(train_s1_set & valid_s1_set))

assert len(train_s1_set & valid_s1_set) == 0

print("✓ No S1 entity appears in both training and validation.")

Overlap: 0
✓ No S1 entity appears in both training and validation.
